# E033 — paired frozen CAFormer-B36 versus CoAtNet exact64 scout

E033 tests one narrow question before spending a full fine-tuning run: does the
ImageNet-pretrained CAFormer-B36 representation transfer to knee MRI better than
the matching ImageNet-pretrained CoAtNet representation when every other choice
is held fixed?

The contract is deliberately paired and fail-closed:

* the fixed 512/128 non-Gold split and exact64 reconstruction are identical for
  both backbones;
* each input is the same 24 safe, same-series 2.5D windows shaped
  `[K, 3, 384, 384]`;
* both backbones stay frozen and receive their own pinned timm preprocessing;
* three identically seeded target-specific MIL heads are trained per backbone;
* selection is made on the fixed non-Gold validation split;
* Gold58 is evaluated exactly once only after the predeclared validation gate;
* passing means only that CAFormer deserves a partial/full fine-tuning run. It
  does not replace the 0.924 CoAtNet submission by itself;
* no checkpoint, feature cache, or competition submission is created.

In [ ]:
from __future__ import annotations

import base64
import gc
import hashlib
import importlib.util
import io
import json
import math
import multiprocessing as mp
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
from pydicom.pixel_data_handlers.util import apply_modality_lut
from sklearn.metrics import roc_auc_score
if importlib.util.find_spec("timm") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "timm>=1.0.15,<2"],
        check=True,
    )
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
CACHE = Path("/kaggle/temp/e033_exact64_cache")
CACHE.mkdir(parents=True, exist_ok=True)
HERE = str(WORK)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]
LAB = TARGETS
IMG, RES, CROP_MM = 336, 384, 140.0
TRAIN_N, VAL_N, K_WINDOWS = 512, 128, 24
K = K_WINDOWS  # default required by the extracted exact64 resample_centers helper
HEAD_DIM = 512
HEAD_SEEDS = [33001, 33002, 33003]
SEED = 31021
SLOTS = [
    ("sagittal_fluid", "Sagittal", 1, 18),
    ("sagittal_other", "Sagittal", 0, 14),
    ("coronal_fluid", "Coronal", 1, 12),
    ("coronal_other", "Coronal", 0, 8),
    ("axial", "Axial", -1, 12),
]
MAXS = sum(slot[3] for slot in SLOTS)
MODEL_SPECS = {
    "coatnet": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k",
    "caformer": "caformer_b36.sail_in22k_ft_in1k_384",
}
MODEL_FEATURE_DIMS = {"coatnet": 1024, "caformer": 768}
EXPECTED_TRAIN_IDS_SHA256 = "0bdc475565a2ff7ea1afc5be9aec69aab0aa615050df796aeac6af4fd2fb051c"
EXPECTED_VALIDATION_IDS_SHA256 = "223ed533e435180acf9f086fff74a12de527977f9bcff7641db7b06d11739565"
EXPECTED_GOLD_IDS_SHA256 = "3918e0c230d6c8d16e073b3c2537d173011b705c71b494493591e6afa5f510ea"
EXPECTED_BASELINE_AUC = 0.9174298724699895
MIN_VALIDATION_GAIN = 0.005
MIN_VALIDATION_TARGET_WINS = 7
MIN_VALIDATION_TARGET_DELTA = -0.05
MIN_GOLD_GAIN = 0.005
MIN_GOLD_TARGET_WINS = 7
MIN_GOLD_TARGET_DELTA = -0.05
MIN_GOLD_BOOTSTRAP_WIN_RATE = 0.70

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = True
print("device", DEVICE, "torch", torch.__version__, "timm", timm.__version__)
print("paired shape", (1, K_WINDOWS, 3, RES, RES), MODEL_SPECS)

In [ ]:
BASELINE_SOURCE = "E029 v3 exact CoAtNet zero-step Gold58 predictions"
BASELINE_PAYLOAD_SHA256 = "7b356156eca446527d89b61cc444ab4622609a63a75eef95d0a3c521fefb99ee"
BASELINE_PAYLOAD_B64 = (
    "UEsDBC0AAAAIAAAAIQCoBgrl//////////8HABQAaWRzLm5weQEAEACAOgAAAAAAADUHAAAAAAAAxZm9jiU1EEZNylNMdkEardp2"
    "/yJiMhDJBkRoxA4iQOxqZkWCeApemBnJRzpd2yQEtSNd3bluu+xyVX31VfU/P7z9/sefvih/lr9u7x6ff3m6fXN3+/btOt/u726/"
    "vn/6+PTwx8/vn949vj747uH358eX8effHj48vvz+atnvv76/+/vuf/99WUspb14+bXzv4/91/J7GN/P6eLaPZ6+feYyz/s0YO/S7"
    "av7r+CI52/jM47tpzqy1fYyv4xnjy3i2j/+bxo8hY9WZavg/U/86dJzGM+as49mi35u++1jT9Nx3gb5t7L2PD3fmtU3rs/VHD+vL"
    "Oep43oMeVXOx8yZ5m/Sdwjfn3cqnvtGS9a+ywTaec377fZVtiYN9yORsvotJMjkLcrC7fW/Rumz9u/bgbIwf5axT1+9Dd+H7mXWO"
    "RXt2jTftd4z/kZutP5i2a+0y7MJz9p81VscYseF4QA7nMa6gD/HC3XXJzdYffddyjkdjgXGaOJjKGbfQ3ToekoO98Qd0JpY+h/8T"
    "61c+D+bZx7Gl9dskb5bMLr3Ihc4Bzifk32z9ORdnBxMdl9hllXzkOSd4Hnsxhg9sF2ssN1t/fJfzgf9H2HsPMtgPO4MP5AR8Hd0c"
    "M/hLzCecKUt//Nv4h+7oA8YZy/GNVs4cx/5jHEA/53/whTtwnGXqb65GjiMnoBe2ZK9Jz21D5wB4BeuIL+4BjOCe7IuZ+ts/sQvc"
    "x9yGPbAbtuR+IhZabzDO+tdyjsFN8zP1N0Y5fxHv5m+uC2bti+7GRvIA8dMlC75gPDw0nqk/Z3AeMGbjC65XiHmP48+c5dDHdaE5"
    "tX2C2MvUv0s25/dZ8PvIA5r2Mv6ZU7muBBPYzzUH+rvmytTfPMw+4TglNs1rzBfMc/2bc9s/zC3cf/gc9jfHRQ7cD5tcnZm4sI+A"
    "DeCh8wTYat8h93D/3EOm/sZ+Yt7YzR1EfYzz7A9XRMfIrc0bnAuNhdn4R7yD1+R993pcu2BzzukeFh9i3tzY+OI4N3Y6H2bp71rE"
    "XOgon9oZvfBjc3zXM8YHeJHzrPkfc90zytIfHMYO5iDYctW8Jrm+I+7ReO5e0KG5rHctYA6VWf9gb/NXdMbG5jtgYNd848F/9Y6r"
    "9sHmEReJo0z+77zv+CYmXQeiK3fhXob3x5bmf+C9n+364FfuoWfp7/7kpn3wXZ5x7sh30d+cxli/hbXmBlWywZ9s/3dOImbxQ3ju"
    "EdZGrmTbmgNTH3IH9ptd+7gGy8Y/96Ccq5BFrnJ/1tyVmHE+XDWPu7J/+J6dI4ixLP3BccffVR2LrYxlrmOsPzJdNxkPuMMe9jIn"
    "yNS/6dt8gLPF913GefeIjOvob5uaZ7vf3Mo5xjLxD8xjvv0dP4w6Er+Lxtira61rqkky3Avhzo0RmfwX/c3/fB77gTmsscC1L7Y0"
    "byBuXA8TJ9QAMX9m6u/cbj81J7VergXBa9+HuQ5237XG99PDns6LGfrbH72OOdiduwAHbdsaPq5vsHv0Cdd+3DF+lxn/+KZ9nzOT"
    "n5C5aJ5xrEpm7AU5doyD7hXE+pj/s/Q3Z7V9fAexzvEc8qZ9Jr7Ldv/Ue4EZVzwpU3/0g3+b+8Qaxe8IuA9sDt5HPGmS5XeCzjPu"
    "RWbpv2gMP/R7KPMD9/Ldy2OuueBczrbknlxvcU7OTC7NxD/XpMYfsG3W/2C1/d5173SxBmxwDHCvluH3h5n5z7wLHHAfi/xlO0d8"
    "2CTHWMf/3A/6wiedF9nD8ZChv/v6nNHYjn6ujZwrrrib68gq+X4nWC/2MxfJ1t84YN90foo9gSn8NjfmN7FUw97mepFTMj9Lf2IQ"
    "LCKWnevd74wYyPeuPbk/5zgw0rpbRqw7MvUHj2O94vc1NYwhx/mffT3uGrdqvxh37J/5/iP2I4lv8rZzF7b3x3nTc80LYp6cJCv6"
    "TZOsLP3NPTkj9naOiud2zWOcxBdir8y9QLDQvNI940z9jb3o5HxofufzGec9B/+JXGHSWvOEHtZk8j/syPmjf5PPokziGXmOe+Nl"
    "vxgDW2M/IObVLP3dn3Tt59qdb/uta173+MgPMaeaI3O3xEsr57vO1v+QfOvh3OQekPkL5wc/5vC/e2y+X9ZEzpnJf9yXonZlnWtT"
    "45hzoOtWZHFO9rG+rn+5R/c8iKVM/V17OP+wp/UgTzMP/wUPzB+8hhoA3X3nsV7OrH9cr2AL5z7bxTmKNeaJzt8ecx8Fv4j9H9dC"
    "5OAM/ZFLPHM+81hjY8QH7sU8oJezHzA/5gXOgc+7DsnU/8omzsXmea2cMT7Wx+aOzIn5we853CvA9zP53641kX+Z8xLb7lX5vS5x"
    "4Dxizmv8sw+Ye9vnsvS370WMN7erkkmOZtzxS+/M/uDaGF9D56v4yrQ/52Ou+zlglvMB92V+xN3s+o7v+DbNJaa4O/uXa48s/V3D"
    "uzY1Jrn/gT3hN8QOOYP9V+1nbulegH+bC/4LUEsDBC0AAAAIAAAAIQBv7nLU//////////8NABQAemVyb19zdGVwLm5weQEAEABg"
    "CwAAAAAAAB0KAAAAAAAAnZX3IxeKAsWzukZIuZKyGrjFNZJwfc8pVEYIIcnKyt4i5KYhRSU9oUEojVu3oUIk1W28ejRllFIpo4TG"
    "jZs8719457dzfv98Tq69i91yd6FxCeOSNf0DYv1iNE3VNM0CDTW11TQDI2PiYnwjvCNj/AP+ty/xDYsNGNtj1/pGBYx1rfnG2mr6"
    "Br9oq6Wq/Z+RVInfC5nCWqht1sb3G/q0nbcOBVad6MvNQ8FaaxaGaDCoThumaVUoK32E7rKHUJBuxurmGMySUaL3s0qIRP3KnhZ7"
    "Ln02j8niKizuNMeyHHs+Dv6JT93nUkS6G3UF0dQpnEzho+vp8I81KxfbcqX0d1T9OgzBYBs2SX+D0tt2FFzbBzX97Shp3YlCn4MQ"
    "TG3GZHMPVJYcxV1DPw6fN6WsbDqOXLgFx4YaKG9O5Z2BNzByM+OyIjMejliAzQczsHNRg0DkaCk2nGzEnWxXHFr1Ayq3H+OQ/3Uk"
    "aI5Co/oYDFVrIfR5EElzn2D9np+YvN+XIakC5vSUY/iNOxv02+FmmEDFpEe4dz8PjzJGYVr/CCs3zmZq/RDahrypVadL/QkWODVX"
    "mftnj+dAgBjvC8VzTkE7NB6vxoPyTLzJ24LjWgvwarQB79P6IPciC93bV1LxmAanuNvxqZw83ftXUHvEmQM//Olj4UqzgoX8HC5H"
    "bXlp3hQbgNM7C45kOFLB2x8P+3px70o/UiKj2Gk5iCnnoqlcuITZ1n40ibegrF00j69qhuyNZhRXNGGo+T264xwgf/UErNs0sHTX"
    "IRyO/AYajiDr+mcsGgxFUW0JCo2fY01/CGO1VrBASZ9TFwcx+IQVyp/+wPGt0/HOMoouU5yo8TUCzWLGrLz8N24tT6TNPh3u/CMH"
    "EoW+zNNThbF1ERa0Dgte3X+MS3GtSLVfhXLhy1BwvwytvbO4M+sW4iJDKVPtQxMbL7ZbKvFqcD30HkXzT01HJuiksrrcgf49lpQz"
    "l2G44UYcvV2Cpuf5mHvNFO8is/H6RLrgj8BwdOk9wHJFW2w8aMArU/tQcFqJ0jUt2F2tjB1a6tzZq4D8NlFKi3YL1tbdwoPMDjiJ"
    "roOt5DK2n4mks3U0J0v04mnsHEY4xTD6YCS1lFLoU+HBERMbFjfM4cPWeOZuk+H2nmqsTH8CubIauGgYsfNGAvVrTbjENIX3bdYy"
    "zKwC9vuacSDMB8UFudB5M4j8L6ehKClNq1OelBcJp/1wPANaQqjaoE7l2rVUy7amwaIYljyz48I0PXZVJvM/jePYJHQFFScVOapD"
    "nLn1GUdz5eD50ot3/gKPnXWA28FQzjdXpeuOZHbNFXDcX1pUKfXn8UE5VPSMIChiWDDOxIWXX+vTkDNQ+s2bYQOibDE0oVbWGcxw"
    "+IjhrX50lzJg3TdDSl5TYezQG8inDyPraCK9JohzUm0jujefg8/haoR8M+VpNUV+Oz2LbVsqEOt+Hn06gVw1w5xClYo8k/Me2utv"
    "IPZ8CsssriFaoQ7P/pak2OIhwcpnGVgf7i/4UqXMM8k/03nQBvG93nSPegibDzH47l6K9LjlNFbOQvqGX/hLoypNznVg2EAWfy6u"
    "Qf8sO0gol8It6glOiIziyd6fWdccyU8NxrRgNJeoP8T4DxNY0aNBw9BpDF0ez9DQhTxwoQdeF32ZY9iEgciVqC0IhGyoJCIH7JCi"
    "sFHg75IH370tqP9qifu7plM38DsuSSZyUp4UnX6A16fbUbAjFq8N/4DIF0JEyIXht8zGnGeHr6dcuTxSglNnrmWGiAs1dTuwNLYW"
    "RR7uyKpXRYnlV4Hj+xDenGfNZbO3orjXgYVu0TRRSuQbewt2xH9Aef9D9N+diTlS6nj6QgrXatbxwn5PbhKywqoef+qs8aBFXSJW"
    "6FxG8vsF2Nc1HXdmZSH50zmkFN7EAttMnP/Yiiq3IzBq9ePT4Ck0K94L01cZ+OLVh+/WSdxX78Kz2im0/h5I31p3iicb0Pp5CUqz"
    "n+KUiSijLDewZasu5ScHc8cjXQ5IDmFvzygun+0V7J4ixYy/hTi+KBqW4p4cb/0e17Nz8TX3GVSsAnj36wc8Mj2MwDwn9OiYouCF"
    "I3s3m3JYYTvanB1ZcEyR4n6O8Pt9DkUOiHLkSQaW5neiUuE5VshosWljEbQqO1FTfADe6mfRWPMEH+28Wf7anDov8jBxsBzum/bg"
    "ns4DFArF4PHWCfTYLUaJpauQvmwFk9IWsOFqIO5duIb4FhWKXUygWaolJXvDucXcjekTruDD6rcw0d0G5XMnoWr8Abv7SlDRVIOM"
    "LUPwiJPmznnebHk9j+eOjfEd58oSl/kMiNqIiA+W7H/+FqIrkjjxd1VOODaIV/l+XJMlgVN5RRDpmoRoiXlUCJpJnYFMvEwL4YQt"
    "njQVSeI2O2du/pcZIyVX0V7qBYJ8H2Nzvg4UrCaxWUWGFvqnMPO2LU1+TOVktxS+63WktpwTLgXNpklYryB9xnG4lClBKM2di5WM"
    "mHjNFq3aYRQUSrItLYxG7QOoLw2lsJIaD+1P4rg0D0qc1aV5tj1DD+lRoySWrQ4reFL4JwbfS2SRfQCdGzowq1yTM+bOgnzpR4Gt"
    "rwRqvkdxX6s9O6+qIc5rKZM71rG/2IjFrxToUuXJHz4KPDXdib2F6sxp1OXnbA9e7P6NU6Yk8sbpMS+P+eFO1Xq6yj2H/XoPfnqp"
    "y2pPM0wMqsSm4ETBcKYVE4U0WZvjDaPDLzDqfBBHexox1/HN2O8HoGxTFQz2LEJ+WS5EK/ow3/MkukVuws4uCGXSbXBv7YBozjos"
    "6Eyg85A+2nW0EeLgiuPN23Cs+S+Y/BaMl+p3MZK6C88MBNzi9AXflZJ4//hc3le5jx3X/0Gyjhga7wVA30YcwrMjmNmxnP+cMMWN"
    "30M423caFbcexAlZVV4ud6VklSnPWilSL8GYwnlq1Ajy5plII+ZsSeKaRkteFPkKlb51rCm9jR3b9Oh9uh9nq/XgM/MIrs3pEnwS"
    "FrDnvRZffxpFqVIoE+SV2b18DSbO6IRw2GQkfPZHmow8yuecxzZzAQpPeNLr3Xzaf9SGi+piTlb5gK4jl2B0+hbmTXTlDQkZSoys"
    "oXhZBEcnxbGrOQjZYz6szRSj7uvHyIx+iEdbN/BM6QR6hYTyyY5QXtqjzkVNNiyclgKfrum883YSh1f6YPHdPmx3uItdVhs46NmK"
    "Nl0rqhuocYmCKYyP5sDy34cE602CKFwAnk+IhvZ5E1ZJPYPbsiS+bBfhQHQgPTUCOf5wCmtUN/C9nANj8pN4fPtqBotZM6dPhYf+"
    "bMCD9a0IbumD+9VPWFIszewESdo9WcSQPbEcqQ+gzTJLvmqrwPA0Ka4SewkVWQXMcz6C6h1HsMkjE4E5rmjq9UKsaDFsNfRQVXQH"
    "Grs6Yb47FUvE6zDa7M2lDRJcGGLCLeI/UzNnjInwBayXVuPQXj0mzZ/OpjthFD3gzFrrmfwvUEsBAi0DLQAAAAgAAAAhAKgGCuU1"
    "BwAAgDoAAAcAAAAAAAAAAAAAAIABAAAAAGlkcy5ucHlQSwECLQMtAAAACAAAACEAb+5y1B0KAABgCwAADQAAAAAAAAAAAAAAgAFu"
    "BwAAemVyb19zdGVwLm5weVBLBQYAAAAAAgACAHAAAADKEQAAAAA="
)
BASELINE_PAYLOAD_BYTES = base64.b64decode(BASELINE_PAYLOAD_B64, validate=True)
if hashlib.sha256(BASELINE_PAYLOAD_BYTES).hexdigest() != BASELINE_PAYLOAD_SHA256:
    raise RuntimeError("embedded E029 baseline payload SHA256 mismatch")
with np.load(io.BytesIO(BASELINE_PAYLOAD_BYTES), allow_pickle=True) as baseline_preflight:
    if baseline_preflight.files != ["ids", "zero_step"]:
        raise RuntimeError("embedded E029 baseline payload schema mismatch")
    if baseline_preflight["zero_step"].shape != (58, len(TARGETS)):
        raise RuntimeError("embedded E029 baseline payload shape mismatch")
print("embedded baseline preflight", len(BASELINE_PAYLOAD_BYTES), BASELINE_PAYLOAD_SHA256)

## Locked exact64 reconstruction and CC0 consensus-label definitions

In [ ]:
def find_root():
    for root in [INPUT / "competitions/rsna-knee-abnormality-detection",
                 INPUT / "rsna-knee-abnormality-detection"]:
        if (root / "train_series.csv").is_file():
            return root
    raise FileNotFoundError("competition mount not found")

def dataset_roots():
    roots = list(INPUT.glob("*")) + list(INPUT.glob("datasets/*/*"))
    return [root for root in roots
            if root.is_dir() and "competitions" not in root.parts and root.name != "datasets"]

def find_input_file(name, preferred=()):
    hits = []
    for root in dataset_roots():
        direct = root / name
        if direct.is_file():
            hits.append(direct)
        hits.extend(path for path in root.glob(f"*/{name}") if path.is_file())
    hits.sort(key=lambda path: (not any(token in str(path).lower() for token in preferred),
                                len(str(path))))
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]

def fluid(value):
    try:
        return int(float(value))
    except Exception:
        return 0

def pick_series(rows, plane, preferred_fluid, used):
    candidates = [row for row in rows
                  if str(row.get("Anatomical_Plane", "")).lower() == plane.lower()
                  and str(row["SeriesInstanceUID"]) not in used]
    if preferred_fluid in (0, 1):
        preferred = [row for row in candidates
                     if fluid(row.get("Fluid_Sensitive", 0)) == preferred_fluid]
        if preferred:
            candidates = preferred
    return candidates[0] if candidates else None

def ordered_records(directory: Path):
    records = []
    for path in sorted(directory.glob("*.dcm")):
        try:
            header = pydicom.dcmread(str(path), stop_before_pixels=True)
            iop = getattr(header, "ImageOrientationPatient", None)
            ipp = getattr(header, "ImagePositionPatient", None)
            if iop is not None and ipp is not None and len(iop) == 6:
                normal = np.cross(np.asarray(iop[:3], float), np.asarray(iop[3:], float))
                position = float(np.dot(np.asarray(ipp, float), normal))
            else:
                position = float(getattr(header, "InstanceNumber", 0) or 0)
            spacing = getattr(header, "PixelSpacing", [0.5, 0.5])
            records.append((position, path, float(spacing[0])))
        except Exception:
            continue
    return sorted(records, key=lambda row: row[0])

def pixels(path):
    dicom = pydicom.dcmread(str(path))
    array = apply_modality_lut(dicom.pixel_array, dicom).astype(np.float32)
    if str(getattr(dicom, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
        array = array.max() - array
    return array

def crop(array01, spacing):
    height, width = array01.shape
    half = max(2, int(round(CROP_MM / max(spacing, 1e-3) / 2)))
    cy, cx = height // 2, width // 2
    value = array01[max(0, cy-half):min(height, cy+half),
                    max(0, cx-half):min(width, cx+half)]
    return cv2.resize(value if value.size else array01, (IMG, IMG), interpolation=cv2.INTER_AREA)

def build_study(uid):
    rows = series_by_uid.get(uid, [])
    volume = np.zeros((64, IMG, IMG), np.uint8)
    mask = np.zeros(64, np.uint8)
    provenance, used, cursor = [], set(), 0
    for slot_index, (slot, plane, preferred_fluid, count) in enumerate(SLOTS):
        row = pick_series(rows, plane, preferred_fluid, used)
        if row is None:
            for slot_position in range(count):
                provenance.append({"StudyInstanceUID": uid, "index": cursor, "slot": slot,
                                   "slot_index": slot_index, "slot_position": slot_position,
                                   "series": None, "raw_index": None, "raw_count": 0,
                                   "valid": 0})
                cursor += 1
            continue

        series_uid = str(row["SeriesInstanceUID"])
        used.add(series_uid)
        records = ordered_records(TRAIN_DCM / uid / series_uid)
        count_raw = len(records)
        lo = int(count_raw * 0.06)
        hi = max(int(count_raw * 0.94) - 1, lo)
        picks = (np.linspace(lo, hi, count).round().astype(int)
                 if count_raw > 1 else np.zeros(count, int))
        arrays = []
        for raw_index in picks:
            try:
                arrays.append(pixels(records[min(int(raw_index), count_raw - 1)][1]))
            except Exception:
                arrays.append(None)
        valid_arrays = [array for array in arrays if array is not None]
        if valid_arrays:
            values = np.concatenate([array.reshape(-1) for array in valid_arrays])
            p02, p98 = np.percentile(values, [2, 98])
        else:
            p02, p98 = 0.0, 1.0

        for slot_position, (raw_index, array) in enumerate(zip(picks, arrays)):
            valid = 0
            if array is not None:
                record = records[min(int(raw_index), count_raw - 1)]
                array01 = np.clip((array - p02) / (p98 - p02 + 1e-6), 0, 1)
                volume[cursor] = np.round(crop(array01, record[2]) * 255).astype(np.uint8)
                mask[cursor] = 1
                valid = 1
            provenance.append({"StudyInstanceUID": uid, "index": cursor, "slot": slot,
                               "slot_index": slot_index, "slot_position": slot_position,
                               "series": series_uid, "raw_index": int(raw_index),
                               "raw_count": count_raw, "valid": valid})
            cursor += 1
    assert cursor == 64
    return volume, mask, pd.DataFrame(provenance)

def safe_candidates(mask, provenance):
    candidates = []
    for center in range(1, len(mask) - 1):
        tri = provenance.iloc[[center - 1, center, center + 1]]
        if (bool(mask[center - 1:center + 2].all())
                and bool(tri["series"].notna().all())
                and int(tri["series"].nunique()) == 1):
            candidates.append(center)
    return candidates

def resample_centers(candidates, count=K):
    if not candidates:
        return []
    indexes = np.linspace(0, len(candidates) - 1, count).round().astype(int)
    return [int(candidates[index]) for index in indexes]

In [ ]:
E021_LABEL_SOURCES = {
    "report_labels_v2.csv": "6f704a7bdb2f894cc49445b19ba7c4378c3f548d3449e00361e10044bee40920",
    "report_labels_gpt56sol.csv": "91678f249ec576d63e16c566b6d7a5c968dfd1af8d1d0721dfde4d0384832add",
    "llm_labels_full.csv": "96444cd3c7d396bae4a71454317d9cbe0689759b5278b30939781e04d0774537",
    "llm_labels_v2.csv": "3c082a987939528a55202028f655bd6f9dc5c9e28eff5058bbae7ef088a9f144",
}

def _e021_source_file(name):
    import hashlib
    from pathlib import Path

    matches = list(Path("/kaggle/input").rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f"E021 expected one {name}, found {len(matches)}")
    path = matches[0]
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != E021_LABEL_SOURCES[name]:
        raise RuntimeError(f"E021 source hash mismatch for {name}: {digest}")
    return path

def _build_e021_labels():
    """Build the predeclared CC0 consensus entirely inside Kaggle."""
    frames = []
    for name in ("report_labels_v2.csv", "report_labels_gpt56sol.csv", "llm_labels_full.csv"):
        frame = pd.read_csv(_e021_source_file(name), usecols=["StudyInstanceUID", *LAB])
        frame["StudyInstanceUID"] = frame["StudyInstanceUID"].astype(str)
        frames.append(frame.set_index("StudyInstanceUID")[LAB].astype(float))

    base_ids = frames[1].index
    if len(base_ids) != 4407 or set(frames[2].index) != set(base_ids):
        raise RuntimeError("E021 label-source UID contract failed")
    values = np.nanmean(
        np.stack([frame.reindex(base_ids).to_numpy() for frame in frames]),
        axis=0,
    )
    consensus = pd.DataFrame(values, index=base_ids, columns=LAB)

    syn = pd.read_csv(
        _e021_source_file("llm_labels_v2.csv"),
        usecols=["StudyInstanceUID", "Synovitis"],
    )
    syn["StudyInstanceUID"] = syn["StudyInstanceUID"].astype(str)
    syn = syn.set_index("StudyInstanceUID")["Synovitis"].astype(float)
    consensus["Synovitis"] = syn.reindex(base_ids)
    consensus = consensus.clip(0.02, 0.98)
    if consensus.shape != (4407, 12) or not np.isfinite(consensus.to_numpy()).all():
        raise RuntimeError(f"E021 consensus contract failed: {consensus.shape}")

    output = os.path.join(HERE, "labels_consensus_cc0.parquet")
    consensus.insert(0, "StudyInstanceUID", consensus.index)
    consensus.to_parquet(output, index=False)
    with open(os.path.join(HERE, "e021_label_manifest.json"), "w") as handle:
        json.dump(
            {
                "rows": 4407,
                "targets": LAB,
                "gold58_audit_macro": 0.8920718993427735,
                "selection": "equal CC0 core; Synovitis from predeclared v2 imputation",
                "source_sha256": E021_LABEL_SOURCES,
            },
            handle,
            indent=2,
        )
    print("E021 consensus labels:", output, consensus.shape, flush=True)
    return output

In [ ]:
def sha256_file(path: Path, chunk_size=8 << 20):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def ids_sha256(ids):
    return hashlib.sha256("\n".join(map(str, ids)).encode()).hexdigest()


def target_aucs(y, predictions):
    values = []
    for index in range(len(TARGETS)):
        if np.unique(y[:, index]).size < 2:
            values.append(np.nan)
        else:
            values.append(roc_auc_score(y[:, index], predictions[:, index]))
    values = np.asarray(values, np.float64)
    return values, float(np.nanmean(values))


def module_sha256(module):
    digest = hashlib.sha256()
    for name, tensor in sorted(module.state_dict().items()):
        array = tensor.detach().cpu().contiguous().numpy()
        digest.update(name.encode())
        digest.update(str(array.shape).encode())
        digest.update(array.tobytes())
    return digest.hexdigest()


ROOT = find_root()
TRAIN_DCM = ROOT / "train_series"
series = pd.read_csv(ROOT / "train_series.csv", dtype={"StudyInstanceUID": str,
                                                        "SeriesInstanceUID": str})
train = pd.read_csv(ROOT / "train.csv", dtype={"StudyInstanceUID": str})
series["StudyInstanceUID"] = series["StudyInstanceUID"].astype(str)
series["SeriesInstanceUID"] = series["SeriesInstanceUID"].astype(str)
train["StudyInstanceUID"] = train["StudyInstanceUID"].astype(str)
series_by_uid = {
    uid: frame.to_dict("records") for uid, frame in series.groupby("StudyInstanceUID")
}

numeric = train[TARGETS].apply(pd.to_numeric, errors="coerce")
gold_mask = numeric.notna().all(axis=1)
gold = train.loc[gold_mask, ["StudyInstanceUID"]].copy()
gold[TARGETS] = numeric.loc[gold_mask].astype(np.float32)
gold_ids = gold["StudyInstanceUID"].tolist()
gold_set = set(gold_ids)
y_gold = gold[TARGETS].to_numpy(np.float32)

all_ids_path = find_input_file("all_ids.npy", ("knee-raptor-corpus",))
extra_ids_path = find_input_file("extra_ids.npy", ("knee-raptor-corpus-ext",))
corpus_ids = [str(value) for value in np.load(all_ids_path, allow_pickle=True).tolist()]
corpus_ids += [str(value) for value in np.load(extra_ids_path, allow_pickle=True).tolist()]
corpus_ids = list(dict.fromkeys(corpus_ids))

label_path = _build_e021_labels()
soft = pd.read_parquet(label_path)
soft["StudyInstanceUID"] = soft["StudyInstanceUID"].astype(str)
soft = soft.set_index("StudyInstanceUID")
candidate_ids = [uid for uid in corpus_ids if uid not in gold_set and uid in soft.index]
rng = np.random.default_rng(SEED)
permutation = rng.permutation(len(candidate_ids))
fixed_train_ids = [candidate_ids[index] for index in permutation[:TRAIN_N]]
fixed_val_ids = [candidate_ids[index] for index in permutation[TRAIN_N:TRAIN_N + VAL_N]]

y_train_soft = soft.loc[fixed_train_ids, TARGETS].to_numpy(np.float32)
y_val_soft = soft.loc[fixed_val_ids, TARGETS].to_numpy(np.float32)
y_val_binary = (y_val_soft >= 0.5).astype(np.float32)
val_class_counts = np.stack([y_val_binary.sum(0), (1 - y_val_binary).sum(0)], axis=1)
val_all_targets_two_classes = bool((val_class_counts.min(1) > 0).all())

with np.load(io.BytesIO(BASELINE_PAYLOAD_BYTES), allow_pickle=True) as baseline_file:
    baseline_ids = [str(value) for value in baseline_file["ids"].tolist()]
    e029_predictions = baseline_file["zero_step"].astype(np.float32)

train_ids_hash = ids_sha256(fixed_train_ids)
validation_ids_hash = ids_sha256(fixed_val_ids)
gold_ids_hash = ids_sha256(gold_ids)
identity_exact = (
    train_ids_hash == EXPECTED_TRAIN_IDS_SHA256
    and validation_ids_hash == EXPECTED_VALIDATION_IDS_SHA256
    and gold_ids_hash == EXPECTED_GOLD_IDS_SHA256
    and baseline_ids == gold_ids
    and e029_predictions.shape == y_gold.shape == (58, len(TARGETS))
)
all_run_ids = fixed_train_ids + fixed_val_ids + gold_ids
identity = {
    "seed": SEED,
    "counts": {"train": TRAIN_N, "validation": VAL_N, "gold": len(gold_ids)},
    "train_ids_sha256": train_ids_hash,
    "validation_ids_sha256": validation_ids_hash,
    "gold_ids_sha256": gold_ids_hash,
    "baseline_payload_sha256": hashlib.sha256(BASELINE_PAYLOAD_BYTES).hexdigest(),
    "all_ids_sha256": sha256_file(Path(all_ids_path)),
    "extra_ids_sha256": sha256_file(Path(extra_ids_path)),
    "identity_exact": identity_exact,
    "validation_class_counts": val_class_counts.astype(int).tolist(),
}
(WORK / "e033_identity.json").write_text(json.dumps(identity, indent=2, sort_keys=True))
print("train/validation/gold", TRAIN_N, VAL_N, len(gold_ids), "identity", identity_exact)
print("validation all targets two classes", val_all_targets_two_classes)

## Build one lossless exact64 cache shared by both frozen backbones

In [ ]:
def delta_x_encode(volume):
    encoded = np.empty_like(volume)
    encoded[:, :, 0] = volume[:, :, 0]
    encoded[:, :, 1:] = volume[:, :, 1:] - volume[:, :, :-1]
    return encoded


def delta_x_decode(encoded):
    return (np.cumsum(encoded, axis=2, dtype=np.uint16) & 255).astype(np.uint8)


def save_lossless_cache(path, volume, mask):
    np.savez_compressed(path, volume_dx=delta_x_encode(volume), mask=mask)


def load_lossless_cache(path):
    with np.load(path) as cached:
        return delta_x_decode(cached["volume_dx"]), cached["mask"]


def build_cache_item(uid):
    cv2.setNumThreads(0)
    started = time.time()
    volume, mask, provenance = build_study(uid)
    candidates = safe_candidates(mask, provenance)
    centers = resample_centers(candidates, count=K_WINDOWS)
    if len(centers) != K_WINDOWS:
        raise RuntimeError(f"{uid}: expected {K_WINDOWS} safe centers")
    cross_series = 0
    zero_slice = 0
    for center in centers:
        tri = provenance.iloc[[center - 1, center, center + 1]]
        zero_slice += int(not bool(mask[center - 1:center + 2].all()))
        cross_series += int(
            bool(tri["series"].isna().any()) or int(tri["series"].nunique()) != 1
        )
    path = CACHE / f"{uid}.npz"
    save_lossless_cache(path, volume, mask)
    decoded, decoded_mask = load_lossless_cache(path)
    pixel_diff = int(np.abs(volume.astype(np.int16) - decoded.astype(np.int16)).max())
    if pixel_diff or not np.array_equal(mask, decoded_mask):
        raise RuntimeError(f"{uid}: cache parity failure")
    return {
        "StudyInstanceUID": uid,
        "provenance": provenance.to_dict("records"),
        "safe_candidates": len(candidates),
        "selected_windows": len(centers),
        "cross_series_windows": cross_series,
        "zero_slice_windows": zero_slice,
        "cache_pixel_diff": pixel_diff,
        "cache_bytes": path.stat().st_size,
        "build_seconds": time.time() - started,
    }


build_workers = min(4, max(1, os.cpu_count() or 1))
cache_started = time.time()
if build_workers == 1:
    cache_records = [build_cache_item(uid) for uid in all_run_ids]
else:
    context = mp.get_context("fork")
    with context.Pool(processes=build_workers) as pool:
        cache_records = list(pool.imap(build_cache_item, all_run_ids, chunksize=1))

cache_paths = {}
provenance_by_uid = {}
cache_rows = []
for record in cache_records:
    uid = record["StudyInstanceUID"]
    provenance_by_uid[uid] = pd.DataFrame(record.pop("provenance"))
    cache_paths[uid] = CACHE / f"{uid}.npz"
    cache_rows.append(record)
cache_metrics = pd.DataFrame(cache_rows)
cache_metrics.to_csv(WORK / "e033_cache_metrics.csv", index=False)
cache_contract_exact = bool(
    (cache_metrics["selected_windows"] == K_WINDOWS).all()
    and int(cache_metrics["cross_series_windows"].sum()) == 0
    and int(cache_metrics["zero_slice_windows"].sum()) == 0
    and int(cache_metrics["cache_pixel_diff"].max()) == 0
)
print("cache minutes/GiB", (time.time() - cache_started) / 60,
      cache_metrics["cache_bytes"].sum() / (1024 ** 3), "exact", cache_contract_exact)

## Extract paired frozen CoAtNet and CAFormer features

In [ ]:
def windows_for_uid(uid):
    volume, mask = load_lossless_cache(cache_paths[uid])
    centers = resample_centers(
        safe_candidates(mask, provenance_by_uid[uid]), count=K_WINDOWS
    )
    if len(centers) != K_WINDOWS:
        raise RuntimeError(f"{uid}: window contract failed")
    windows = np.stack([
        volume[[center - 1, center, center + 1]].astype(np.float32) / 255.0
        for center in centers
    ])
    tensor = F.interpolate(
        torch.from_numpy(windows), size=(RES, RES), mode="bilinear", align_corners=False
    )
    return tensor, centers


@torch.inference_mode()
def encode_backbone(model_key, ids):
    architecture = MODEL_SPECS[model_key]
    model = timm.create_model(
        architecture, pretrained=True, num_classes=0, in_chans=3, global_pool="avg"
    )
    model.eval()
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    num_features = int(model.num_features)
    expected_features = MODEL_FEATURE_DIMS[model_key]
    state_sha256 = module_sha256(model)
    config = timm.data.resolve_model_data_config(model)
    input_size = tuple(int(value) for value in config["input_size"])
    mean = torch.tensor(config["mean"], dtype=torch.float32).view(3, 1, 1)
    std = torch.tensor(config["std"], dtype=torch.float32).view(3, 1, 1)
    if input_size != (3, RES, RES) or num_features != expected_features:
        raise RuntimeError(
            f"{architecture}: expected input/features {(3, RES, RES)}/{expected_features}, "
            f"got {input_size}/{num_features}"
        )
    model.to(DEVICE)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)
    features = np.empty((len(ids), K_WINDOWS, expected_features), np.float16)
    started = time.time()
    for study_index, uid in enumerate(ids):
        tensor, _centers = windows_for_uid(uid)
        tensor = (tensor - mean) / std
        chunks = []
        for start in range(0, K_WINDOWS, 6):
            values = tensor[start:start + 6].to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                encoded = model(values)
            chunks.append(encoded.float().cpu())
        study_features = torch.cat(chunks, dim=0)
        if tuple(study_features.shape) != (K_WINDOWS, expected_features):
            raise RuntimeError(f"{architecture}: bad feature shape {tuple(study_features.shape)}")
        features[study_index] = study_features.numpy().astype(np.float16)
        if (study_index + 1) % 50 == 0:
            print(model_key, "encoded", study_index + 1, "/", len(ids), flush=True)
    elapsed = time.time() - started
    feature_float = features.astype(np.float32)
    sample = torch.from_numpy(feature_float[: min(64, len(features))])
    normalized = F.normalize(sample, dim=-1)
    within_cosine = torch.einsum("bkf,blf->bkl", normalized, normalized)
    off_diagonal = ~torch.eye(K_WINDOWS, dtype=torch.bool)[None]
    mean_within_cosine = float(within_cosine.masked_select(off_diagonal).mean())
    record = {
        "model_key": model_key,
        "architecture": architecture,
        "parameter_count": int(parameter_count),
        "num_features": num_features,
        "input_size": list(input_size),
        "mean": list(map(float, config["mean"])),
        "std": list(map(float, config["std"])),
        "state_sha256": state_sha256,
        "feature_shape": list(features.shape),
        "feature_finite": bool(np.isfinite(feature_float).all()),
        "feature_std": float(feature_float.std()),
        "mean_within_study_cosine": mean_within_cosine,
        "elapsed_seconds": elapsed,
        "windows_per_second": float(len(ids) * K_WINDOWS / elapsed),
        "peak_gpu_gib": (
            torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 3)
            if DEVICE.type == "cuda" else 0.0
        ),
    }
    del model, sample, normalized, within_cosine, feature_float
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print(model_key, record)
    return features, record


non_gold_ids = fixed_train_ids + fixed_val_ids
paired_features = {}
feature_records = []
for model_key in ("coatnet", "caformer"):
    paired_features[model_key], record = encode_backbone(model_key, non_gold_ids)
    feature_records.append(record)
pd.DataFrame(feature_records).to_csv(WORK / "e033_feature_metrics.csv", index=False)

## Train identical three-seed target-specific MIL heads

In [ ]:
class TargetWindowMIL(nn.Module):
    def __init__(self, input_dim, hidden_dim=HEAD_DIM, targets=len(TARGETS), dropout=0.15):
        super().__init__()
        self.projection = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(256, targets),
        )
        self.classifier_weight = nn.Parameter(torch.empty(targets, hidden_dim))
        self.classifier_bias = nn.Parameter(torch.zeros(targets))
        nn.init.trunc_normal_(self.classifier_weight, std=0.02)

    def forward(self, features, return_attention=False):
        hidden = self.projection(features)
        attention = torch.softmax(self.attention(hidden), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", attention, hidden)
        logits = (
            pooled * self.classifier_weight.unsqueeze(0)
        ).sum(-1) + self.classifier_bias
        if return_attention:
            return logits, attention.transpose(1, 2)
        return logits


@torch.inference_mode()
def predict_head(model, features, batch_size=64, return_attention=False):
    model.eval()
    predictions = []
    attentions = []
    for start in range(0, len(features), batch_size):
        values = torch.from_numpy(features[start:start + batch_size]).to(DEVICE).float()
        if return_attention:
            logits, attention = model(values, return_attention=True)
            attentions.append(attention.cpu())
        else:
            logits = model(values)
        predictions.append(torch.sigmoid(logits).cpu())
    prediction_array = torch.cat(predictions).numpy().astype(np.float32)
    if not return_attention:
        return prediction_array
    attention_array = torch.cat(attentions).numpy().astype(np.float32)
    return prediction_array, attention_array


pos_weight = torch.tensor(
    np.clip((1 - y_train_soft.mean(0)) / np.clip(y_train_soft.mean(0), 0.03, 1), 1, 10),
    dtype=torch.float32,
    device=DEVICE,
)
loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


def train_one_head(model_key, features, seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    train_features = torch.from_numpy(features[:TRAIN_N])
    train_targets = torch.from_numpy(y_train_soft)
    loader = DataLoader(
        TensorDataset(train_features, train_targets),
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
        num_workers=0,
        drop_last=False,
    )
    input_dim = int(features.shape[-1])
    model = TargetWindowMIL(input_dim=input_dim).to(DEVICE)
    initial_hash = module_sha256(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=80, eta_min=1e-6
    )
    best_auc = -np.inf
    best_epoch = -1
    best_state = None
    stale = 0
    history = []
    for epoch in range(80):
        model.train()
        total_loss = 0.0
        for batch_features, batch_targets in loader:
            values = batch_features.to(DEVICE).float()
            targets = batch_targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(values)
            loss = loss_function(logits, targets)
            if not torch.isfinite(loss):
                raise RuntimeError(f"{model_key}/{seed}: non-finite loss")
            loss.backward()
            gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            if not torch.isfinite(gradient_norm):
                raise RuntimeError(f"{model_key}/{seed}: non-finite gradient")
            optimizer.step()
            total_loss += float(loss.item()) * len(values)
        scheduler.step()
        validation_predictions = predict_head(model, features[TRAIN_N:TRAIN_N + VAL_N])
        _target_values, validation_auc = target_aucs(y_val_binary, validation_predictions)
        epoch_loss = total_loss / TRAIN_N
        history.append({
            "model_key": model_key,
            "seed": seed,
            "epoch": epoch,
            "loss": epoch_loss,
            "validation_macro_auc": validation_auc,
        })
        if validation_auc > best_auc + 1e-6:
            best_auc = validation_auc
            best_epoch = epoch
            best_state = {
                name: value.detach().cpu().clone() for name, value in model.state_dict().items()
            }
            stale = 0
        else:
            stale += 1
        if stale >= 12 and epoch >= 20:
            break
    if best_state is None:
        raise RuntimeError(f"{model_key}/{seed}: no finite best state")
    model.load_state_dict(best_state, strict=True)
    validation_predictions, validation_attention = predict_head(
        model, features[TRAIN_N:TRAIN_N + VAL_N], return_attention=True
    )
    final_hash = module_sha256(model)
    record = {
        "model_key": model_key,
        "seed": seed,
        "best_epoch": best_epoch,
        "validation_macro_auc": best_auc,
        "input_dim": input_dim,
        "hidden_dim": HEAD_DIM,
        "initial_sha256": initial_hash,
        "final_sha256": final_hash,
        "updated": initial_hash != final_hash,
    }
    del model, loader, train_features, train_targets
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return best_state, validation_predictions, validation_attention, history, record


head_states = {model_key: [] for model_key in MODEL_SPECS}
validation_predictions_by_seed = {model_key: [] for model_key in MODEL_SPECS}
validation_attention_by_seed = {model_key: [] for model_key in MODEL_SPECS}
head_history = []
head_records = []
for model_key in ("coatnet", "caformer"):
    for head_seed in HEAD_SEEDS:
        state, predictions, attention, history, record = train_one_head(
            model_key, paired_features[model_key], head_seed
        )
        head_states[model_key].append(state)
        validation_predictions_by_seed[model_key].append(predictions)
        validation_attention_by_seed[model_key].append(attention)
        head_history.extend(history)
        head_records.append(record)
        print("head", record, flush=True)
pd.DataFrame(head_history).to_csv(WORK / "e033_head_history.csv", index=False)

## Apply the locked non-Gold validation gate

In [ ]:
validation_ensemble = {
    model_key: np.mean(validation_predictions_by_seed[model_key], axis=0).astype(np.float32)
    for model_key in MODEL_SPECS
}
validation_target_auc = {}
validation_macro_auc = {}
attention_rows = []
for model_key in ("coatnet", "caformer"):
    target_values, macro = target_aucs(y_val_binary, validation_ensemble[model_key])
    validation_target_auc[model_key] = target_values
    validation_macro_auc[model_key] = macro
    ensemble_attention = np.mean(validation_attention_by_seed[model_key], axis=0)
    entropy = -np.sum(
        ensemble_attention * np.log(np.clip(ensemble_attention, 1e-8, 1.0)), axis=-1
    ) / math.log(K_WINDOWS)
    for target_index, target in enumerate(TARGETS):
        attention_rows.append({
            "model_key": model_key,
            "target": target,
            "mean_normalized_entropy": float(entropy[:, target_index].mean()),
            "across_study_std": float(ensemble_attention[:, target_index].std()),
            "nondegenerate": bool(ensemble_attention[:, target_index].std() > 1e-5),
        })
attention_metrics = pd.DataFrame(attention_rows)
attention_metrics.to_csv(WORK / "e033_attention_metrics.csv", index=False)

validation_delta = validation_target_auc["caformer"] - validation_target_auc["coatnet"]
validation_gain = validation_macro_auc["caformer"] - validation_macro_auc["coatnet"]
validation_target_wins = int(np.sum(validation_delta > 0))
validation_min_target_delta = float(np.nanmin(validation_delta))
validation_prediction_correlation = float(np.corrcoef(
    validation_ensemble["coatnet"].reshape(-1),
    validation_ensemble["caformer"].reshape(-1),
)[0, 1])
validation_target_metrics = pd.DataFrame({
    "target": TARGETS,
    "coatnet_auc": validation_target_auc["coatnet"],
    "caformer_auc": validation_target_auc["caformer"],
    "delta_caformer_minus_coatnet": validation_delta,
})
validation_target_metrics.to_csv(WORK / "e033_validation_target_metrics.csv", index=False)
np.savez_compressed(
    WORK / "e033_validation_predictions.npz",
    ids=np.asarray(fixed_val_ids),
    y_true=y_val_binary,
    coatnet=validation_ensemble["coatnet"],
    caformer=validation_ensemble["caformer"],
)

required_gates = {
    "identity_exact": bool(identity_exact),
    "validation_all_targets_two_classes": bool(val_all_targets_two_classes),
    "cache_contract_exact": bool(cache_contract_exact),
    "model_shapes_exact": bool(all(
        row["input_size"] == [3, RES, RES]
        and row["num_features"] == MODEL_FEATURE_DIMS[row["model_key"]]
        and row["feature_shape"] == [
            TRAIN_N + VAL_N, K_WINDOWS, MODEL_FEATURE_DIMS[row["model_key"]]
        ]
        for row in feature_records
    )),
    "features_finite_nondegenerate": bool(all(
        row["feature_finite"] and row["feature_std"] > 0.01 for row in feature_records
    )),
    "all_heads_updated": bool(all(row["updated"] for row in head_records)),
    "attention_nondegenerate": bool(attention_metrics["nondegenerate"].all()),
}
pre_gold_gates = {
    "caformer_validation_gain": bool(validation_gain >= MIN_VALIDATION_GAIN),
    "caformer_target_wins": bool(validation_target_wins >= MIN_VALIDATION_TARGET_WINS),
    "caformer_no_severe_target_drop": bool(
        validation_min_target_delta >= MIN_VALIDATION_TARGET_DELTA
    ),
}
pre_gold_pass = bool(all(required_gates.values()) and all(pre_gold_gates.values()))
print("validation", validation_macro_auc, "gain", validation_gain,
      "wins/min", validation_target_wins, validation_min_target_delta,
      "correlation", validation_prediction_correlation, "open Gold", pre_gold_pass)

## Open Gold58 once only if the validation gate passed

In [ ]:
def paired_bootstrap(y, baseline, challenger, repeats=2000):
    rng = np.random.default_rng(SEED + 33)
    differences = []
    for _ in range(repeats):
        indexes = rng.integers(0, len(y), size=len(y))
        per_target = []
        for target_index in range(len(TARGETS)):
            labels = y[indexes, target_index]
            if np.unique(labels).size < 2:
                continue
            baseline_auc = roc_auc_score(labels, baseline[indexes, target_index])
            challenger_auc = roc_auc_score(labels, challenger[indexes, target_index])
            per_target.append(challenger_auc - baseline_auc)
        if per_target:
            differences.append(float(np.mean(per_target)))
    differences = np.asarray(differences, np.float64)
    return {
        "repeats_valid": int(len(differences)),
        "mean_delta": float(differences.mean()),
        "win_rate": float(np.mean(differences > 0)),
        "ci_low": float(np.quantile(differences, 0.025)),
        "ci_high": float(np.quantile(differences, 0.975)),
    }


gold_metrics = None
gold_signal_gates = {
    "caformer_gold_gain": False,
    "caformer_gold_target_wins": False,
    "caformer_gold_no_severe_target_drop": False,
    "paired_bootstrap_signal": False,
}
gold_evaluation_count = 0
if pre_gold_pass:
    gold_evaluation_count = 1
    gold_features = {}
    gold_feature_records = []
    for model_key in ("coatnet", "caformer"):
        gold_features[model_key], record = encode_backbone(model_key, gold_ids)
        record["split"] = "gold58"
        gold_feature_records.append(record)

    gold_predictions = {}
    for model_key in ("coatnet", "caformer"):
        predictions = []
        for state in head_states[model_key]:
            head = TargetWindowMIL(input_dim=MODEL_FEATURE_DIMS[model_key]).to(DEVICE)
            head.load_state_dict(state, strict=True)
            predictions.append(predict_head(head, gold_features[model_key]))
            del head
        gold_predictions[model_key] = np.mean(predictions, axis=0).astype(np.float32)

    e029_target_auc, e029_macro_auc = target_aucs(y_gold, e029_predictions)
    coatnet_gold_target_auc, coatnet_gold_macro_auc = target_aucs(
        y_gold, gold_predictions["coatnet"]
    )
    caformer_gold_target_auc, caformer_gold_macro_auc = target_aucs(
        y_gold, gold_predictions["caformer"]
    )
    if abs(e029_macro_auc - EXPECTED_BASELINE_AUC) > 1e-12:
        raise RuntimeError(f"E029 baseline drift: {e029_macro_auc}")
    gold_delta = caformer_gold_target_auc - coatnet_gold_target_auc
    gold_gain = caformer_gold_macro_auc - coatnet_gold_macro_auc
    gold_target_wins = int(np.sum(gold_delta > 0))
    gold_min_target_delta = float(np.nanmin(gold_delta))
    bootstrap = paired_bootstrap(
        y_gold, gold_predictions["coatnet"], gold_predictions["caformer"]
    )
    gold_signal_gates = {
        "caformer_gold_gain": bool(gold_gain >= MIN_GOLD_GAIN),
        "caformer_gold_target_wins": bool(gold_target_wins >= MIN_GOLD_TARGET_WINS),
        "caformer_gold_no_severe_target_drop": bool(
            gold_min_target_delta >= MIN_GOLD_TARGET_DELTA
        ),
        "paired_bootstrap_signal": bool(
            bootstrap["win_rate"] >= MIN_GOLD_BOOTSTRAP_WIN_RATE
        ),
    }
    gold_metrics = {
        "evaluation_count": 1,
        "e029_finetuned_coatnet_macro_auc": e029_macro_auc,
        "paired_frozen_coatnet_macro_auc": coatnet_gold_macro_auc,
        "paired_frozen_caformer_macro_auc": caformer_gold_macro_auc,
        "delta_caformer_minus_frozen_coatnet": gold_gain,
        "target_wins": gold_target_wins,
        "min_target_delta": gold_min_target_delta,
        "paired_bootstrap": bootstrap,
    }
    pd.DataFrame({
        "target": TARGETS,
        "e029_finetuned_coatnet_auc": e029_target_auc,
        "paired_frozen_coatnet_auc": coatnet_gold_target_auc,
        "paired_frozen_caformer_auc": caformer_gold_target_auc,
        "delta_caformer_minus_frozen_coatnet": gold_delta,
    }).to_csv(WORK / "e033_gold58_target_metrics.csv", index=False)
    np.savez_compressed(
        WORK / "e033_gold58_predictions.npz",
        ids=np.asarray(gold_ids),
        y_true=y_gold,
        e029_finetuned_coatnet=e029_predictions,
        paired_frozen_coatnet=gold_predictions["coatnet"],
        paired_frozen_caformer=gold_predictions["caformer"],
    )
    print("Gold58", gold_metrics, "gates", gold_signal_gates)
else:
    print("Gold58 skipped because the fixed non-Gold validation gate failed")

## Write the fail-closed decision and compact evidence

In [ ]:
all_required_pass = bool(all(required_gates.values()))
all_pre_gold_pass = bool(all(pre_gold_gates.values()))
all_gold_signal_pass = bool(all(gold_signal_gates.values()))
if not all_required_pass:
    decision = "STOP_E033_REQUIRED_GATE_FAILED"
elif not all_pre_gold_pass:
    decision = "STOP_E033_PRE_GOLD_FAILED"
elif not all_gold_signal_pass:
    decision = "STOP_E033_GOLD_SIGNAL_FAILED"
else:
    decision = "PASS_E033_BACKBONE_SIGNAL_NO_SUBMISSION"

summary = {
    "decision": decision,
    "required_gates": required_gates,
    "pre_gold_gates": pre_gold_gates,
    "gold_signal_gates": gold_signal_gates,
    "identity": identity,
    "input": {
        "shape": [1, K_WINDOWS, 3, RES, RES],
        "exact64_slots": [list(slot) for slot in SLOTS],
        "safe_same_series_windows": True,
        "cross_series_window_count": int(cache_metrics["cross_series_windows"].sum()),
        "zero_slice_window_count": int(cache_metrics["zero_slice_windows"].sum()),
    },
    "models": feature_records,
    "training": {
        "mode": "paired_frozen_backbones_fresh_target_window_mil",
        "head_seeds": HEAD_SEEDS,
        "shared_hidden_dim": HEAD_DIM,
        "head_records": head_records,
        "selection_split": "fixed_non_gold_validation",
    },
    "validation": {
        "coatnet_macro_auc": validation_macro_auc["coatnet"],
        "caformer_macro_auc": validation_macro_auc["caformer"],
        "delta_caformer_minus_coatnet": validation_gain,
        "target_wins": validation_target_wins,
        "min_target_delta": validation_min_target_delta,
        "prediction_correlation": validation_prediction_correlation,
    },
    "gold58": {
        "gate_passed": pre_gold_pass,
        "evaluation_count": gold_evaluation_count,
        "metrics": gold_metrics,
    },
    "submission_created": False,
    "checkpoint_exported": False,
    "feature_cache_exported": False,
}
(WORK / "e033_summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True))
gate_rows = []
for group, gates in (
    ("required", required_gates),
    ("pre_gold", pre_gold_gates),
    ("gold_signal", gold_signal_gates),
):
    for gate, passed in gates.items():
        gate_rows.append({"group": group, "gate": gate, "passed": bool(passed)})
pd.DataFrame(gate_rows).to_csv(WORK / "e033_gates.csv", index=False)

shutil.rmtree(CACHE, ignore_errors=True)
del paired_features
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

for forbidden in ("submission.csv",):
    if (WORK / forbidden).exists():
        raise RuntimeError(f"forbidden artifact exists: {forbidden}")
for path in WORK.iterdir():
    if path.suffix.lower() in {".pt", ".pth", ".ckpt", ".safetensors"}:
        raise RuntimeError(f"forbidden checkpoint artifact exists: {path.name}")
print("FINAL", decision)
print(json.dumps(summary["validation"], indent=2))
print(json.dumps(summary["gold58"], indent=2))